# NB10 v2 -- car-only vs. full-class confidence_threshold sweep

Single tile: `dop20_32_473_5525_1_he` (Darmstadt-Hbf-adjacent rail
corridor -- converging tracks, no station building, industrial/warehouse
buildings with solar roofs, roads, scattered trees, two car parking lots,
one train with car-carrier wagons).

Sweeps `confidence_threshold` = `[0.01, 0.02, 0.05, 0.1, 0.2, 0.3, 0.5, 0.7]`
(8 values, wider than v1's 0.05-0.5) in **two modes**:
- Run A: `words=["car"]` (car-only, same as v1)
- Run B: full 7-class list (building, road, tracks, train, car, tree, grass)

`prob_thd=0.1` and window params (`slide_crop=1024`, `slide_stride=768`)
held fixed throughout, same as v1 -- only `confidence_threshold` and the
prompt-set vary. 16 full sliding-window reruns total (8 thresholds x 2
prompt sets).

`tracks` is included in Run B despite two prior NB09 tiles (`475_5550`,
`473_5524`) where a standalone tracks/railway/platform class never
produced a reliable detection -- both failures were driven by
tracks/railway getting confused with a **platform** structure in strong
sun. This tile has no platform, so that specific confusion mechanism
can't occur here; `tracks` gets a genuinely fresh shot on tile-specific
grounds (not just "try again anyway").

Deliverables are primarily **numeric** (pixel/blob counts, per-class
trend plots, a model-derived car/train overlap analysis), not a
per-threshold image gallery -- only one reference overlay image per run
mode is rendered. See `docs/project/2026-07-26_v1_nb10v2-car-confthd-devlog.md`
for the full reasoning trail.


## 1 — Environment setup

In [ ]:
import os

!wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O /tmp/miniconda_installer.sh
!bash /tmp/miniconda_installer.sh -b -p /tmp/miniconda

os.environ.pop("PYTHONPATH", None)
os.environ["PATH"] = "/tmp/miniconda/bin:" + os.environ["PATH"]

!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r
!conda --version

In [ ]:
!/tmp/miniconda/bin/conda create -n segearth python=3.10 -y

In [ ]:
!conda run -n segearth pip install torch==2.4.0 torchvision==0.19.0 -q

In [ ]:
!conda run -n segearth pip install openmim -q
!conda run -n segearth mim install "mmcv==2.2.0" -q
!conda run -n segearth pip install "mmsegmentation==1.2.2" -q

In [ ]:
%%bash
source /tmp/miniconda/bin/activate segearth
python - << 'EOF'
import pathlib
f = pathlib.Path("/tmp/miniconda/envs/segearth/lib/python3.10/site-packages/mmseg/__init__.py")
f.write_text(f.read_text().replace("MMCV_MAX = '2.2.0'", "MMCV_MAX = '2.3.0'"))
print("Patched MMCV_MAX \u2192 2.3.0")
EOF
pip install numpy==1.26.4 -q

In [ ]:
%%bash
source /tmp/miniconda/bin/activate segearth
python - << 'EOF'
import mmcv; print("MMCV:", mmcv.__version__)
from mmseg.structures import SegDataSample; print("MMSEG OK")
import torch; print("CUDA:", torch.cuda.is_available())
EOF

## 2 — Clone our fork

In [ ]:
import subprocess, os
from pathlib import Path

REPO = Path("/tmp/SegEarth-OV-3")

if REPO.exists():
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
    print(f"Updated \u2192 {REPO}")
else:
    subprocess.run(
        ["git", "clone", "--depth=1",
         "https://github.com/HarishDeepak/rg-segearth-ov3", str(REPO)],
        check=True)
    print(f"Cloned \u2192 {REPO}")

os.chdir(REPO)
!conda run -n segearth pip install -r requirements.txt -q

## 3 -- Inference: car-only + full-class, confidence_threshold sweep (16 reruns)

In [ ]:
%%bash
export MPLBACKEND=Agg
export PYTHONUNBUFFERED=1
source /tmp/miniconda/bin/activate segearth
cd /tmp/SegEarth-OV-3

python - << 'PYEOF'
import sys, json, torch, torch.nn.functional as F
import numpy as np
from pathlib import Path
from PIL import Image

sys.stdout.reconfigure(line_buffering=True)

DEVICE   = "cuda"
OUT_DIR  = Path("/kaggle/working/output"); OUT_DIR.mkdir(parents=True, exist_ok=True)
PRED_DIR = OUT_DIR / "preds"; PRED_DIR.mkdir(parents=True, exist_ok=True)
BG_IDX = 255

PROB_THD     = 0.1
SLIDE_CROP   = 1024
SLIDE_STRIDE = 768
CONF_THD_SWEEP = [0.01, 0.02, 0.05, 0.1, 0.2, 0.3, 0.5, 0.7]

STEM = "dop20_32_473_5525_1_he"

FULL_CLASS_WORDS = [
    "building, warehouse, industrial shed, solar panel roof",
    "road, paved street",
    "railway tracks, rail lines, ballast trackbed, gravel between rails",
    "train, railway wagon, rolling stock",
    "car, vehicle",
    "tree",
    "grass, low vegetation",
]
FULL_CLASS_DISPLAY = ["building", "road", "tracks", "train", "car", "tree", "grass"]
FULL_CLASS_COLORS = [
    [0, 0, 255],
    [60, 60, 60],
    [128, 128, 0],
    [255, 140, 0],
    [255, 255, 0],
    [0, 200, 0],
    [0, 255, 255],
]
CAR_ONLY_WORDS = ["car"]

def find_tile(stem):
    hits = sorted(Path("/kaggle/input").rglob(f"{stem}.jpg"))
    return hits[0] if hits else None

img_path = find_tile(STEM)
print(f"Resolved {STEM} -> {img_path}", flush=True)
if img_path is None:
    print("ERROR: tile not found under /kaggle/input.", flush=True)
    raise SystemExit(1)

from config_local import SAM3_CHECKPOINT
from sam3 import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor

print("Loading SAM3...", flush=True)
model = build_sam3_image_model(
    bpe_path="./sam3/assets/bpe_simple_vocab_16e6.txt.gz",
    checkpoint_path=SAM3_CHECKPOINT, device=DEVICE)
model.eval()
for p in model.parameters(): p.requires_grad = False
print(f"GPU: {torch.cuda.get_device_name(0)}", flush=True)

def make_processor(conf_thd):
    return Sam3Processor(model, confidence_threshold=conf_thd, device=DEVICE)

def cache_text(processor, words):
    cache = []
    with torch.no_grad():
        for word in words:
            te = model.backbone.forward_text([word], device=DEVICE)
            cache.append({k: v.cpu() for k, v in te.items()})
    return cache

def collect_class_scores(processor, state, h, w, te_cache, n_classes, device):
    logits = torch.zeros((n_classes, h, w), device=device)
    for cls_idx, te_cpu in enumerate(te_cache):
        processor.reset_all_prompts(state)
        for k, v in te_cpu.items(): state["backbone_out"][k] = v.to(device)
        state["geometric_prompt"] = model._get_dummy_prompt()
        processor._forward_grounding(state)
        scores = torch.zeros((h, w), device=device)
        if state.get("masks_logits") is not None and state["masks_logits"].shape[0] > 0:
            for i in range(state["masks_logits"].shape[0]):
                il = state["masks_logits"][i].squeeze()
                if il.shape != (h, w):
                    il = F.interpolate(il.view(1,1,*il.shape), size=(h,w),
                                       mode="bilinear", align_corners=False).squeeze()
                scores = torch.max(scores, il * state["object_score"][i])
        sem = state["semantic_mask_logits"].squeeze()
        if sem.shape != (h, w):
            sem = F.interpolate(sem.view(1,1,*sem.shape), size=(h,w),
                                mode="bilinear", align_corners=False).squeeze()
        scores = torch.max(scores, sem) * state["presence_score"]
        logits[cls_idx] = torch.max(logits[cls_idx], scores)
    return logits

def make_gaussian_kernel(h, w, dev):
    sy, sx = h/4.0, w/4.0
    y = torch.arange(h, device=dev).float() - (h-1)/2.0
    x = torch.arange(w, device=dev).float() - (w-1)/2.0
    return torch.exp(-y[:,None]**2/(2*sy**2)) * torch.exp(-x[None,:]**2/(2*sx**2))

def run_sliding_window(img_arr, words, processor, crop_size, stride):
    te_cache = cache_text(processor, words)
    n_cls = len(words)
    H_full, W_full = img_arr.shape[:2]

    h_grids = max(H_full - crop_size + stride - 1, 0) // stride + 1
    w_grids = max(W_full - crop_size + stride - 1, 0) // stride + 1
    total = h_grids * w_grids

    gauss_k = make_gaussian_kernel(crop_size, crop_size, DEVICE)
    acc     = torch.zeros(n_cls, H_full, W_full, device=DEVICE)
    wt_mat  = torch.zeros(H_full, W_full, device=DEVICE)

    for hi in range(h_grids):
        for wi in range(w_grids):
            y1 = hi*stride;  x1 = wi*stride
            y2 = min(y1+crop_size, H_full);  x2 = min(x1+crop_size, W_full)
            y1 = max(y2-crop_size, 0);       x1 = max(x2-crop_size, 0)

            crop_pil = Image.fromarray(img_arr[y1:y2, x1:x2])
            h_c, w_c = y2-y1, x2-x1

            with torch.no_grad(), torch.autocast("cuda", dtype=torch.bfloat16):
                state = processor.set_image(crop_pil)
                l = collect_class_scores(processor, state, h_c, w_c, te_cache, n_cls, DEVICE).float()

            g = gauss_k[:h_c, :w_c]
            acc[:, y1:y2, x1:x2] += l * g.unsqueeze(0)
            wt_mat[y1:y2, x1:x2] += g

            done = hi*w_grids + wi + 1
            print(f"    crop {done}/{total}", flush=True)

    return acc / wt_mat.unsqueeze(0)

def finalize(prob_map, prob_thd, bg_idx=BG_IDX):
    seg = prob_map.argmax(0)
    seg[prob_map.max(0)[0] < prob_thd] = bg_idx
    return seg.cpu().numpy()

manifest = []

def save_pred(mode, conf_thd, seg, logits=None):
    tag = f"{mode}_conf{conf_thd}"
    np.save(str(PRED_DIR / f"{STEM}_{tag}.npy"), seg.astype(np.uint8))
    if logits is not None:
        np.save(str(PRED_DIR / f"{STEM}_{tag}_logits.npy"), logits.cpu().numpy().astype(np.float16))
    manifest.append(dict(stem=STEM, mode=mode, tag=tag, prob_thd=PROB_THD, conf_thd=conf_thd,
                          slide_stride=SLIDE_STRIDE, slide_crop=SLIDE_CROP))
    print(f"  Saved pred: {tag}", flush=True)

img_arr = np.array(Image.open(img_path).convert("RGB"))
img_size = (img_arr.shape[1], img_arr.shape[0])

print(f"\n=== Run A: car-only ===", flush=True)
for ct in CONF_THD_SWEEP:
    print(f"  [confidence_threshold={ct}] car-only", flush=True)
    proc = make_processor(ct)
    logits = run_sliding_window(img_arr, CAR_ONLY_WORDS, proc, SLIDE_CROP, SLIDE_STRIDE)
    seg = finalize(logits, PROB_THD)
    save_pred("caronly", ct, seg)

print(f"\n=== Run B: full class list ===", flush=True)
for ct in CONF_THD_SWEEP:
    print(f"  [confidence_threshold={ct}] full-class ({len(FULL_CLASS_WORDS)} classes)", flush=True)
    proc = make_processor(ct)
    logits = run_sliding_window(img_arr, FULL_CLASS_WORDS, proc, SLIDE_CROP, SLIDE_STRIDE)
    seg = finalize(logits, PROB_THD)
    save_pred("fullclass", ct, seg, logits=logits)

(PRED_DIR / f"{STEM}_meta.json").write_text(json.dumps(dict(
    img_size=img_size, full_class_display=FULL_CLASS_DISPLAY,
    full_class_colors=FULL_CLASS_COLORS, full_class_words=FULL_CLASS_WORDS)))

(OUT_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2))
print(f"\nWrote manifest.json with {len(manifest)} entries", flush=True)
PYEOF


## 4 -- Metrics + rendering (GPU-free)

Numeric analysis is the primary deliverable (per-class pixel-count trend
plots, car-only vs full-class comparison, model-derived car/train overlap
analysis) plus one reference overlay image per run mode.


In [ ]:
import json
import numpy as np
from pathlib import Path
from PIL import Image
from scipy.ndimage import label
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

OUT_DIR  = Path("/kaggle/working/output")
PRED_DIR = OUT_DIR / "preds"
BG_IDX = 255

manifest = json.loads((OUT_DIR / "manifest.json").read_text())
STEM = manifest[0]["stem"]
meta = json.loads((PRED_DIR / f"{STEM}_meta.json").read_text())
FULL_CLASS_DISPLAY = meta["full_class_display"]
FULL_CLASS_COLORS  = np.array(meta["full_class_colors"], dtype=np.uint8)
CAR_CLASS_IDX      = FULL_CLASS_DISPLAY.index("car")
TRAIN_CLASS_IDX    = FULL_CLASS_DISPLAY.index("train")

img_arr = np.array(Image.open([p for p in Path("/kaggle/input").rglob(f"{STEM}.jpg")][0]).convert("RGB"))

def to_rgb_binary(seg, color=(255, 255, 0)):
    out = np.full((*seg.shape, 3), 30, dtype=np.uint8)
    out[seg == 0] = color
    return out

def to_rgb_multiclass(seg, color_map, bg_idx=BG_IDX):
    out = np.zeros((*seg.shape, 3), dtype=np.uint8)
    safe = np.where(seg == bg_idx, 0, seg)
    out[:] = color_map[np.clip(safe, 0, len(color_map)-1)]
    out[seg == bg_idx] = [30, 30, 30]
    return out

def render_overlay(seg, out_path, title, color_fn, display_labels=None, color_map=None, alpha=0.6):
    fig = plt.figure(figsize=(20, 12))
    ax = fig.subplots(1, 2)
    ax[0].imshow(img_arr); ax[0].axis("off")
    ax[0].set_title(f"{STEM}.jpg", fontsize=12, fontweight="bold")
    ax[1].imshow(img_arr)
    ax[1].imshow(color_fn(seg), alpha=alpha)
    ax[1].axis("off")
    ax[1].set_title(title, fontsize=12, fontweight="bold")
    if display_labels is not None:
        legend_elements = [Patch(facecolor=c / 255.0, edgecolor="black", label=n)
                            for n, c in zip(display_labels, color_map)]
        plt.subplots_adjust(left=0.01, right=0.99, top=0.95, bottom=0.15, wspace=0.01)
        fig.legend(handles=legend_elements, loc="lower center", bbox_to_anchor=(0.5, 0.05),
                   frameon=False, fontsize=9, ncol=min(7, len(display_labels)), prop={"weight": "bold"})
    fig.savefig(str(out_path), dpi=200, bbox_inches="tight")
    plt.close(fig)
    print(f"  Rendered: {out_path.name}", flush=True)

# Run A: car-only metrics
caronly_rows = []
for entry in [e for e in manifest if e["mode"] == "caronly"]:
    seg = np.load(str(PRED_DIR / f"{STEM}_{entry['tag']}.npy"))
    car_mask = seg == 0
    pix = int(car_mask.sum())
    _, blobs = label(car_mask)
    caronly_rows.append((entry["conf_thd"], pix, blobs))
caronly_rows.sort()
print("Run A (car-only):", caronly_rows)

ref_ct = 0.1
ref_seg = np.load(str(PRED_DIR / f"{STEM}_caronly_conf{ref_ct}.npy"))
render_overlay(ref_seg, OUT_DIR / f"{STEM}_caronly_conf{ref_ct}_reference.png",
               f"car only, confidence_threshold={ref_ct}", to_rgb_binary)

cts, pix, blobs = zip(*caronly_rows)
fig, ax1 = plt.subplots(figsize=(9, 5))
ax1.plot(cts, pix, "o-", color="tab:blue", label="car pixel count")
ax1.set_xlabel("confidence_threshold")
ax1.set_ylabel("car pixel count", color="tab:blue")
ax1.tick_params(axis="y", labelcolor="tab:blue")
ax2 = ax1.twinx()
ax2.plot(cts, blobs, "s--", color="tab:red", label="car blob count")
ax2.set_ylabel("car blob count", color="tab:red")
ax2.tick_params(axis="y", labelcolor="tab:red")
plt.title(f"{STEM}: car-only run -- car detection vs confidence_threshold")
fig.tight_layout()
fig.savefig(str(OUT_DIR / f"{STEM}_runA_caronly_trend.png"), dpi=150)
plt.close(fig)

# Run B: full-class metrics
fullclass_rows = {cls: [] for cls in FULL_CLASS_DISPLAY}
fullclass_car_rows = []
for entry in [e for e in manifest if e["mode"] == "fullclass"]:
    seg = np.load(str(PRED_DIR / f"{STEM}_{entry['tag']}.npy"))
    ct = entry["conf_thd"]
    for cls_idx, cls_name in enumerate(FULL_CLASS_DISPLAY):
        pix = int((seg == cls_idx).sum())
        fullclass_rows[cls_name].append((ct, pix))
    fullclass_car_rows.append((ct, int((seg == CAR_CLASS_IDX).sum())))
for cls in fullclass_rows:
    fullclass_rows[cls].sort()
fullclass_car_rows.sort()
print("Run B (full-class) per-class pixel counts:")
for cls, rows in fullclass_rows.items():
    print(f"  {cls}: {rows}")

ref_seg_b = np.load(str(PRED_DIR / f"{STEM}_fullclass_conf{ref_ct}.npy"))
render_overlay(ref_seg_b, OUT_DIR / f"{STEM}_fullclass_conf{ref_ct}_reference.png",
               f"full class list, confidence_threshold={ref_ct}",
               lambda s: to_rgb_multiclass(s, FULL_CLASS_COLORS),
               display_labels=FULL_CLASS_DISPLAY, color_map=FULL_CLASS_COLORS)

fig, ax = plt.subplots(figsize=(9, 6))
for cls_idx, cls_name in enumerate(FULL_CLASS_DISPLAY):
    rows = fullclass_rows[cls_name]
    cts_b, pix_b = zip(*rows)
    color = FULL_CLASS_COLORS[cls_idx] / 255.0
    ax.plot(cts_b, pix_b, "o-", color=color, label=cls_name)
ax.set_xlabel("confidence_threshold")
ax.set_ylabel("pixel count")
ax.set_title(f"{STEM}: full-class run -- per-class pixel count vs confidence_threshold")
ax.legend()
fig.tight_layout()
fig.savefig(str(OUT_DIR / f"{STEM}_runB_fullclass_trend.png"), dpi=150)
plt.close(fig)

# Car-only vs full-class comparison
cts_a, pix_a, _ = zip(*caronly_rows)
cts_b2, pix_b2 = zip(*fullclass_car_rows)
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(cts_a, pix_a, "o-", color="tab:blue", label="car alone (Run A)")
ax.plot(cts_b2, pix_b2, "s--", color="tab:orange", label="car within full class list (Run B)")
ax.set_xlabel("confidence_threshold")
ax.set_ylabel("car pixel count")
ax.set_title(f"{STEM}: car pixel count, alone vs. within full class list")
ax.legend()
fig.tight_layout()
fig.savefig(str(OUT_DIR / f"{STEM}_car_alone_vs_fullclass.png"), dpi=150)
plt.close(fig)
print("\nRun A vs Run B car pixel counts (conf_thd, alone, in-full-class):")
for (ct, a, _), (_, b) in zip(caronly_rows, fullclass_car_rows):
    print(f"  {ct}: alone={a}  in-full-class={b}  diff={a-b}")

# Car/train overlap region -- model-derived from saved logits
print("\n=== Car/train overlap analysis (model-derived, no manual bbox) ===", flush=True)
overlap_rows = []
for entry in [e for e in manifest if e["mode"] == "fullclass"]:
    ct = entry["conf_thd"]
    logits_path = PRED_DIR / f"{STEM}_{entry['tag']}_logits.npy"
    if not logits_path.exists():
        continue
    logits = np.load(str(logits_path)).astype(np.float32)
    car_logit = logits[CAR_CLASS_IDX]
    train_logit = logits[TRAIN_CLASS_IDX]
    car_thresh = np.percentile(car_logit, 99)
    train_thresh = np.percentile(train_logit, 99)
    overlap_mask = (car_logit >= car_thresh) & (train_logit >= train_thresh)
    n_overlap = int(overlap_mask.sum())
    if n_overlap > 0:
        ys, xs = np.where(overlap_mask)
        bbox = (int(xs.min()), int(ys.min()), int(xs.max()), int(ys.max()))
    else:
        bbox = None
    overlap_rows.append((ct, n_overlap, bbox))
    print(f"  conf_thd={ct}: overlap_pixels={n_overlap}  bbox={bbox}", flush=True)

if any(n > 0 for _, n, _ in overlap_rows):
    best_ct, best_n, best_bbox = max(overlap_rows, key=lambda r: r[1])
    print(f"\nLargest car/train overlap at confidence_threshold={best_ct}: "
          f"{best_n} px, bbox={best_bbox}", flush=True)
    if best_bbox is not None:
        x1, y1, x2, y2 = best_bbox
        pad = 100
        x1, y1 = max(0, x1 - pad), max(0, y1 - pad)
        x2, y2 = min(img_arr.shape[1], x2 + pad), min(img_arr.shape[0], y2 + pad)
        crop_seg = ref_seg_b[y1:y2, x1:x2]
        crop_img = img_arr[y1:y2, x1:x2]
        fig = plt.figure(figsize=(10, 10))
        ax = fig.subplots(1, 2)
        ax[0].imshow(crop_img); ax[0].axis("off")
        ax[0].set_title(f"car/train overlap region crop ({STEM})")
        ax[1].imshow(crop_img)
        ax[1].imshow(to_rgb_multiclass(crop_seg, FULL_CLASS_COLORS), alpha=0.6)
        ax[1].axis("off")
        ax[1].set_title(f"full-class seg, conf_thd={ref_ct}")
        fig.savefig(str(OUT_DIR / f"{STEM}_car_train_overlap_crop.png"), dpi=150, bbox_inches="tight")
        plt.close(fig)
        print(f"  Rendered: {STEM}_car_train_overlap_crop.png (model-derived region)", flush=True)
else:
    print("\nNo meaningful car/train overlap region found at any threshold -- "
          "argmax cleanly separates 'car' and 'train' everywhere on this tile.", flush=True)

print(f"\nDone. Rendered images + trend plots + overlap analysis for {STEM}.", flush=True)
